# Lab 7: Bayesian Regression Workflow

**Phân tích dữ liệu Bayesian - IUH**

Một bảng số liệu chiều cao và cân nặng có thể được đọc theo nhiều cách. Ta có thể tìm một đường hồi quy “tốt nhất” theo bình phương tối thiểu và dừng lại ở đó. Nhưng ta cũng có thể hỏi nhiều hơn: độ dốc thật sự còn bất định đến đâu, dự đoán cho một người mới nên rộng đến mức nào, và làm sao biết mô hình đang dùng có đang bỏ sót điều gì trong dữ liệu?

Lab 7 đi theo hướng thứ hai. Nó dùng bài toán hồi quy như một nơi để gom các bước của Bayesian workflow vào cùng một dòng chảy: đặt mô hình, suy luận posterior, tạo dự đoán, rồi quay lại kiểm tra mô hình.


In [ ]:
# Dữ liệu chiều cao-cân nặng được giữ trong notebook để workflow hồi quy Bayesian tự chạy không phụ thuộc file ngoài.
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["figure.figsize"] = (10, 5)

height = np.array([14.36, 14.48, 14.53, 14.52, 14.35, 14.31, 14.44, 14.23, 14.32,
                   14.57, 14.28, 14.36, 14.50, 14.52, 14.28, 14.13, 14.54, 14.60,
                   14.86, 14.28, 14.09, 14.20, 14.50, 14.02, 14.45])
weight = np.array([14.49, 14.64, 14.76, 14.70, 14.55, 14.46, 14.59, 14.26, 14.37,
                   14.77, 14.37, 14.50, 14.63, 14.64, 14.39, 14.23, 14.68, 14.73,
                   14.93, 14.40, 14.22, 14.31, 14.65, 14.10, 14.55])


## 1. Từ một đường fit đến cả một workflow dự đoán

Điểm bắt đầu của lab vẫn là một ý tưởng quen thuộc: nếu chiều cao tăng, cân nặng có xu hướng thay đổi theo cách nào? Nhưng thay vì chỉ săn tìm một đường thẳng trung tâm, ta xem hệ số chặn, hệ số dốc và độ nhiễu là những đại lượng chưa biết và để dữ liệu cập nhật niềm tin về chúng.

Để giữ trọng tâm ở workflow, lab này bắt đầu với một mô hình hồi quy Gaussian tối giản. Nhờ vậy, người học có thể tập trung vào luồng suy nghĩ quan trọng hơn: so sánh với baseline cổ điển, đọc posterior của hệ số, chuyển posterior thành dự đoán cho quan sát mới, rồi dùng residual để tự hỏi mô hình hiện tại đã đủ tốt hay chưa.


## 2. Baseline: least squares as a reference point

Least squares được đặt ở đầu không phải để thay thế Bayesian regression, mà để làm một mốc tham chiếu quen thuộc. Khi đã có điểm mốc này, người học sẽ dễ thấy posterior mean của hệ số hồi quy gần nó ở đâu và khác nó ở chỗ nào.

### Ý nghĩa của bài tập

Ý nghĩa của bước baseline là giúp sinh viên không xem hồi quy Bayesian như một thứ hoàn toàn tách rời thống kê cổ điển. Nhiều ý tưởng quen thuộc như đường fit trung tâm hay phần dư vẫn còn đó, chỉ là Bayes bổ sung thêm cấu trúc bất định quanh các đại lượng ấy.

Đây là cách nhập môn rất hiệu quả cho workflow hồi quy: hiểu điểm ước lượng cổ điển trước, rồi mới mở rộng sang phân phối hậu nghiệm.


### Đề bài nhắc lại

**Lab 7 - bài toán gốc.** Cho bảng dữ liệu gồm 25 cặp $(X,Y)$ biểu diễn chiều cao và cân nặng. Câu hỏi trung tâm là: liệu có thể dự đoán cân nặng của một người dựa vào chiều cao của họ không? Phần này bắt đầu bằng đường hồi quy bình phương tối thiểu như một mốc tham chiếu trước khi chuyển sang mô hình Bayes.


In [ ]:
# Trước hết lấy least squares làm mốc tham chiếu, rồi mới so với posterior từ mô hình Bayesian Gaussian.
X = np.column_stack([np.ones_like(height), height - height.mean()])
beta_hat = np.linalg.inv(X.T @ X) @ X.T @ weight
residuals = weight - X @ beta_hat
sigma_hat = np.sqrt(np.sum(residuals**2) / (len(weight) - X.shape[1]))

print(f"OLS intercept at centered x = {beta_hat[0]:.4f}")
print(f"OLS slope = {beta_hat[1]:.4f}")
print(f"Residual sd estimate = {sigma_hat:.4f}")


## 3. Bayesian linear regression with normal prior on coefficients

Ta đặt prior
$$\beta \sim N(m_0, V_0), \qquad y \mid \beta \sim N(X\beta, \sigma^2 I),$$
với $m_0=(\bar y, 0)$ và $V_0=\mathrm{diag}(1, 1)$. Khi $\sigma$ xem như đã biết, posterior của $\beta$ vẫn là Gaussian:
$$V_n^{-1}=V_0^{-1}+\frac{X^TX}{\sigma^2}, \qquad m_n = V_n\left(V_0^{-1}m_0 + \frac{X^Ty}{\sigma^2}\right).$$

### Ý nghĩa của bài tập

Đây là trái tim của cả lab. Bài này cho thấy trong hồi quy Bayesian, hệ số chặn và hệ số dốc không còn là những con số cố định được ước lượng xong là hết, mà là các biến ngẫu nhiên có posterior riêng sau khi nhìn dữ liệu.

Ý nghĩa sâu hơn là prior bắt đầu đóng vai trò regularization một cách minh bạch. Thay vì “phạt” hệ số theo kiểu thuật toán, ta diễn đạt niềm tin ban đầu về kích thước hợp lý của hệ số rồi để dữ liệu điều chỉnh niềm tin đó.


### Đề bài nhắc lại

Từ cùng bộ dữ liệu chiều cao-cân nặng, hãy xây dựng một mô hình hồi quy Bayes cho mối liên hệ giữa $X$ và $Y$, thay vì chỉ dùng least squares. Phần solution này diễn giải lại các ví dụ trong lab gốc theo workflow hồi quy Bayes hiện đại.


In [ ]:
# Khi sigma được xem là đã biết, posterior của vector hệ số vẫn là Gaussian và có công thức đóng.
m0 = np.array([weight.mean(), 0.0])
V0 = np.diag([1.0, 1.0])
V0_inv = np.linalg.inv(V0)
Vn = np.linalg.inv(V0_inv + (X.T @ X) / sigma_hat**2)
mn = Vn @ (V0_inv @ m0 + (X.T @ weight) / sigma_hat**2)

posterior_sd = np.sqrt(np.diag(Vn))
print(f"Posterior mean intercept = {mn[0]:.4f} +/- {posterior_sd[0]:.4f}")
print(f"Posterior mean slope = {mn[1]:.4f} +/- {posterior_sd[1]:.4f}")


## 4. Posterior predictive reasoning

Posterior predictive là bước biến suy luận về tham số thành suy luận về dữ liệu tương lai. Đây là lúc Bayesian regression cho thấy giá trị thực hành rõ nhất: ta không chỉ học về đường hồi quy, mà còn học về độ bất định khi dự đoán một quan sát mới.

### Ý nghĩa của bài tập

Ý nghĩa của bước này là giúp sinh viên phân biệt hai loại bất định thường bị trộn lẫn: bất định về giá trị trung bình có điều kiện và bất định của chính quan sát mới quanh giá trị trung bình đó. Posterior predictive gộp cả hai lớp bất định lại trong một đối tượng duy nhất.

Một khi hiểu được bước này, người học sẽ thấy vì sao Bayesian workflow đặc biệt mạnh trong các bài toán dự báo chứ không chỉ trong ước lượng tham số.


### Đề bài nhắc lại

Sau khi đã có phân phối hậu nghiệm của các hệ số hồi quy, mục tiêu tiếp theo là dùng posterior để dự đoán cân nặng cho các giá trị chiều cao mới, đồng thời giữ lại bất định của mô hình và của nhiễu quan sát.


In [ ]:
# Lấy mẫu từ posterior của beta để chuyển bất định tham số thành bất định dự đoán cho quan sát mới.
rng = np.random.default_rng(2024)
beta_samples = rng.multivariate_normal(mn, Vn, size=4000)
height_grid = np.linspace(height.min(), height.max(), 100)
X_grid = np.column_stack([np.ones_like(height_grid), height_grid - height.mean()])
mu_samples = X_grid @ beta_samples.T
pred_samples = rng.normal(mu_samples, sigma_hat)

mean_line = mu_samples.mean(axis=1)
interval_90 = np.quantile(pred_samples, [0.05, 0.95], axis=1)

plt.scatter(height, weight, color='black', alpha=0.7, label='data')
plt.plot(height_grid, mean_line, color='tab:blue', label='posterior mean fit')
plt.fill_between(height_grid, interval_90[0], interval_90[1], color='tab:blue', alpha=0.2, label='90% predictive interval')
plt.xlabel('height')
plt.ylabel('weight')
plt.title('Bayesian regression fit and predictive uncertainty')
plt.legend()
plt.show()


### Interpretation

Posterior mean của hệ số dốc dương cho thấy cân nặng tăng khi chiều cao tăng. Nhưng điều quan trọng hơn là predictive interval: nó nhắc ta rằng ngay cả khi biết khá rõ đường hồi quy trung bình, cân nặng của một cá thể mới vẫn có độ bất định đáng kể quanh đường đó.


## 5. Posterior predictive check and reflection

Phần residual check đặt ngay sau posterior predictive để nhấn mạnh rằng suy luận Bayesian chưa hoàn tất khi đã có posterior đẹp. Mô hình vẫn phải được chất vấn xem nó có bỏ sót cấu trúc nào trong dữ liệu hay không.

### Ý nghĩa của bài tập

Ý nghĩa của bước này là biến model criticism thành một phần bắt buộc của workflow, chứ không phải một phụ lục tùy chọn. Nếu residual còn có xu hướng hay cấu trúc, ta học được điều gì đó về giới hạn của mô hình tuyến tính Gaussian hiện tại.

Bài này vì thế dạy một thói quen rất lành mạnh: posterior chỉ đáng tin khi ta cũng kiểm tra được chất lượng của mô hình đã sinh ra posterior đó.


### Đề bài nhắc lại

Phần cuối của lab yêu cầu không chỉ fit hồi quy mà còn kiểm tra xem mô hình dự đoán có hợp lý với dữ liệu quan sát hay không. Vì vậy solution bổ sung posterior predictive check và phần phản tư về chất lượng mô hình.


In [ ]:
# Residual check giúp xem mô hình tuyến tính Gaussian còn bỏ sót cấu trúc nào trong dữ liệu hay không.
fitted = X @ mn
resid_post = weight - fitted
print(f"Mean posterior residual = {resid_post.mean():.6f}")
print(f"Residual sd around posterior mean fit = {resid_post.std(ddof=1):.4f}")

plt.scatter(fitted, resid_post, color='tab:red')
plt.axhline(0, color='black', linestyle='--')
plt.xlabel('fitted values')
plt.ylabel('residuals')
plt.title('Residual check for the Bayesian regression fit')
plt.show()


Nếu phần dư có cấu trúc cong mạnh hoặc phương sai thay đổi theo chiều cao, mô hình tuyến tính Gaussian sẽ là chưa đủ. Khi đó ta nên cân nhắc biến đổi biến, thêm predictor, hoặc dùng một mô hình phương sai không hằng.


## 6. Connection back to the original lab

Notebook gốc còn giới thiệu nhiều ví dụ về posterior rời rạc, mô hình nhị thức, rejection sampling, Gibbs sampling và logistic regression. Những phần đó hữu ích như tài liệu tham khảo phương pháp. Tuy nhiên, để solution notebook nhất quán và dễ học, tôi gom trọng tâm vào một chu trình hoàn chỉnh: mô hình hóa, posterior, dự đoán và kiểm tra mô hình trong bài hồi quy tuyến tính.


## 7. Conceptual interpretation questions

1. Vì sao least squares chỉ là điểm tham chiếu chứ chưa phải là toàn bộ suy luận Bayesian?  
2. Sự khác biệt giữa bất định của hệ số hồi quy và bất định dự đoán cho một quan sát mới là gì?  
3. Khi residual plot cho thấy cấu trúc, điều đó nói gì về giả định mô hình?  
4. Trong hồi quy Bayesian, prior trên hệ số dốc đóng vai trò regularization như thế nào?


## 8. Final takeaway

Lab 7 nên được đọc như một bài về workflow hơn là một cuộc thi cài thư viện. Bản chất của Bayesian regression là mô tả mối liên hệ giữa dữ liệu và các tham số chưa biết, rồi chuyển posterior đó thành dự đoán và kiểm tra mô hình một cách có ý nghĩa.


## 9. Lecture references

Nếu muốn dùng lecture để đi lại đúng workflow của Lab 7, nên đọc lại các phần sau:

- [Bài 4.1: Bayesian Linear Regression - Mô hình Sinh dữ liệu](../../contents/vi/chapter04/_posts/2025-01-02-04_01_linear_regression_generative.md): **Bài 2** và **Bài 3**: đọc lại câu chuyện sinh dữ liệu của hồi quy tuyến tính trước khi nhìn vào công thức posterior cho hệ số.
- [Bài 4.2: Priors for Regression - Chọn Prior có Nguyên tắc](../../contents/vi/chapter04/_posts/2025-01-02-04_02_priors_for_regression.md): **Bài 3**: đặc biệt cần khi muốn hiểu prior trên intercept và slope đang regularize mô hình ra sao, thay vì chỉ xem đó là tham số mặc định.
- [Bài 4.3: Posterior Inference với PyMC - Từ Theory đến Practice](../../contents/vi/chapter04/_posts/2025-01-02-04_03_posterior_inference_pymc.md): **Bài 3**: đọc lại nếu muốn nối lời giải đóng dạng trong notebook với cách suy luận hậu nghiệm tổng quát hơn trong thực hành.
- [Bài 4.4: Model Checking và Prediction - Đảm bảo Model Tốt](../../contents/vi/chapter04/_posts/2025-01-02-04_04_model_checking_prediction.md): **Bài 4** và **Bài 5**: đọc lại khi muốn phân biệt posterior predictive với fitted mean, và hiểu residual check đang kiểm tra giả định gì của mô hình.
- [Bài 10.4: Bayesian Workflow - Putting It All Together](../../contents/vi/chapter10/_posts/2025-01-04-10_04_bayesian_workflow_synthesis.md): toàn bộ **Lab 7**: đọc lại để thấy baseline, posterior, predictive và model criticism không phải các mảnh rời rạc mà là một workflow thống nhất.
